# GameTheory 04h (Lean) - Prix de l'anarchie : la borne atomique 2x

**Navigation** : [<< 04h-Routing (track principal)](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb) | [Index](README.md)

Ce carnet est le **compagnon formel** de [04h-Routing (Python)](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb). Le carnet Python *mesure* le prix de l'anarchie sur une instance de Pigou non-atomique et obtient la borne 4/3 de Roughgarden-Tardos (2002) ; ici, nous **prouvons** le resultat correspondant dans le modele **atomique** : n joueurs discrets, couts lineaires entiers, et la borne elementaire **2x** portee par le potentiel de Rosenthal (1973).

Tout le contenu formel vit dans le module lake `game_theory_lean/CongestionGames/PoA.lean` — ce carnet l'importe, l'execute et l'interprete ; il ne redemontre rien a la main. Le modele vit entierement dans `Nat` : aucune fraction, aucun flottant.

## 1. Le modele : charges, couts, potentiel

Un jeu de congestion atomique lineaire, c'est :
- un ensemble fini d'aretes `E` (routes, segments) ;
- une **pente** `a e` par arete : sous charge `x`, la latence de `e` vaut `a e * x` ;
- `n` joueurs, chacun choisissant un ensemble fini d'aretes.

Trois quantites s'en deduisent, toutes entieres.

In [1]:
import CongestionGames.PoA

open CongestionGames

-- Charge de l'arete e : nombre de joueurs dont la strategie contient e
#check @load
-- Cout individuel du joueur i : somme des latences de ses aretes
#check @playerCost
-- Cout social (latence totale) : sum_e a_e * x_e^2
#check @socialCost
-- Potentiel de Rosenthal DOUBLE : sum_e a_e * x_e * (x_e + 1)
#check @potential

import CongestionGames.PoA

open CongestionGames

-- Charge de l'arete e : nombre de joueurs dont la strategie contient e
#check @load
──────▶  @load : {E : Type} → [DecidableEq E] → {n : ℕ} → (Fin n → Finset E) → E → ℕ
-- Cout individuel du joueur i : somme des latences de ses aretes
#check @playerCost
──────▶  @playerCost : {E : Type} → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → Fin n → ℕ
-- Cout social (latence totale) : sum_e a_e * x_e^2
#check @socialCost
──────▶  @socialCost : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ
-- Potentiel de Rosenthal DOUBLE : sum_e a_e * x_e * (x_e + 1)
#check @potential
──────▶  @potential : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ
--% env 0

Raw input:
{"cmd": "import CongestionGames.PoA\n\nopen CongestionGames\n\n-- Charge de l'arete e : nombre de joueurs dont la strategie contient e\n#check @load\n-- Cout individuel du joueur i : somme des latences de ses aretes\n#check @playerCost\n-- Cout social (latence totale) : sum_e a_e * x_e^2\n#check @socialCost\n-- Potentiel de Rosenthal DOUBLE : sum_e a_e * x_e * (x_e + 1)\n#check @potential"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@load : {E : Type} → [DecidableEq E] → {n : ℕ} → (Fin n → Finset E) → E → ℕ"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@playerCost : {E : Type} → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → Fin n → ℕ"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "@socialCost : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 6},
   "data":
   "@potential : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ"}],
 "env": 0}

### Lecture du resultat

La definition singuliere est `potential` : le potentiel classique de Rosenthal vaut `Phi = sum_e a_e x_e^2 / 2` — une **fraction**. Pour rester dans `Nat`, le module utilise la forme **doublee** `Phi' = sum_e a_e x_e (x_e + 1)`. Ce +1 dans le second facteur n'est pas une commodite gratuite : c'est lui qui rend le potentiel **exact** pour les deviations unilaterales, comme le montre l'identite suivante.

## 2. L'identite d'exactitude : pourquoi le facteur 2

Quand le joueur `i` devie unilateralement de sa strategie courante vers `s`, deux choses changent : son propre cout, et le potentiel. Le theoreme central du module dit qu'ils changent **exactement ensemble**, avec un facteur 2 :

In [2]:
-- Identite d'exactitude du potentiel double
#check @potential_exchange

/-- Signature explicite : pour toute deviation unilaterale,
    Phi'(sigma') + 2 * cout_i(sigma) = Phi'(sigma) + 2 * cout_i(sigma'). -/
example : ∀ {E : Type} [Fintype E] [DecidableEq E] {n : ℕ} (a : E → ℕ)
    (σ : Fin n → Finset E) (i : Fin n) (s : Finset E),
    potential a (Function.update σ i s) + 2 * playerCost a σ i
      = potential a σ + 2 * playerCost a (Function.update σ i s) i :=
  potential_exchange

-- Identite d'exactitude du potentiel double
#check @potential_exchange
──────▶  @potential_exchange : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)
  (σ : Fin n → Finset E) (i : Fin n) (s : Finset E),
  potential a (Function.update σ i s) + 2 * playerCost a σ i =
    potential a σ + 2 * playerCost a (Function.update σ i s) i

/-- Signature explicite : pour toute deviation unilaterale,
    Phi'(sigma') + 2 * cout_i(sigma) = Phi'(sigma) + 2 * cout_i(sigma'). -/
example : ∀ {E : Type} [Fintype E] [DecidableEq E] {n : ℕ} (a : E → ℕ)
    (σ : Fin n → Finset E) (i : Fin n) (s : Finset E),
    potential a (Function.update σ i s) + 2 * playerCost a σ i
      = potential a σ + 2 * playerCost a (Function.update σ i s) i :=
  potential_exchange
--% env 1

Raw input:
{"cmd": "-- Identite d'exactitude du potentiel double\n#check @potential_exchange\n\n/-- Signature explicite : pour toute deviation unilaterale,\n    Phi'(sigma') + 2 * cout_i(sigma) = Phi'(sigma) + 2 * cout_i(sigma'). -/\nexample : \u2200 {E : Type} [Fintype E] [DecidableEq E] {n : \u2115} (a : E \u2192 \u2115)\n    (\u03c3 : Fin n \u2192 Finset E) (i : Fin n) (s : Finset E),\n    potential a (Function.update \u03c3 i s) + 2 * playerCost a \u03c3 i\n      = potential a \u03c3 + 2 * playerCost a (Function.update \u03c3 i s) i :=\n  potential_exchange", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@potential_exchange : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)\n  (σ : Fin n → Finset E) (i : Fin n) (s : Finset E),\n  potential a (Function.update σ i s) + 2 * playerCost a σ i =\n    potential a σ + 2 * playerCost a (Function.update σ i s) i"}],
 "env": 1}

### Lecture du resultat

Verifions la mecanique sur une arete `e` que la deviation **gagne** (`e` dans `s`, pas dans l'ancienne strategie) : sa charge passe de `x` a `x + 1`. Cote potentiel : `a x (x+1)` devient `a (x+1) (x+2)`, un accroissement de `2 a (x+1)`. Cote cout du joueur : son cout d'usage de `e` passe de 0 a `a (x+1)`. Le potentiel double croit exactement **deux fois** le cout du deviant. Le calcul est symetrique sur une arete **perdue**, et neutre sur une arete commune — c'est ce que la preuve `potential_exchange` formalise arete par arete, puis somme.

**Consequence immediate** : une deviation qui diminue strictement le cout de `i` diminue strictement le potentiel. Les equilibres de Nash sont donc exactement les profils *localement minimaux* du potentiel.

## 3. Le theoreme : la chaine de Rosenthal

La borne 2x s'obtient en trois inegalites elementaires, chacune portee par un lemme du module :

In [3]:
-- Maillon 1 : x^2 <= x (x+1), arete par arete
#check @socialCost_le_potential
-- Maillon 2 : minimalite du potentiel sur les profils jouables
#check @minPotential_isNash
-- Maillon 3 : x (x+1) <= 2 x^2, arete par arete
#check @potential_le_two_socialCost
-- La chaine complete
#check @cost_minPotential_le_two

-- Preuves sans sorry ni axiome ad hoc
#print axioms CongestionGames.potential_exchange
#print axioms CongestionGames.minPotential_isNash
#print axioms CongestionGames.cost_minPotential_le_two

-- Maillon 1 : x^2 <= x (x+1), arete par arete
#check @socialCost_le_potential
──────▶  @socialCost_le_potential : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)
  (σ : Fin n → Finset E), socialCost a σ ≤ potential a σ
-- Maillon 2 : minimalite du potentiel sur les profils jouables
#check @minPotential_isNash
──────▶  @minPotential_isNash : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)
  (S : Fin n → Finset (Finset E)) {σh : Fin n → Finset E},
  PlaysIn S σh → (∀ (τ : Fin n → Finset E), PlaysIn S τ → potential a σh ≤ potential a τ) → IsNash a S σh
-- Maillon 3 : x (x+1) <= 2 x^2, arete par arete
#check @potential_le_two_socialCost
──────▶  @potential_le_two_socialCost : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)
  (σ : Fin n → Finset E), potential a σ ≤ 2 * socialCost a σ
-- La chaine complete
#check @cost_minPotential_le_two
──────▶  @cost_minPotential_le_two : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)
  (S : Fin n → Finset (Finset E)) {σh τ : Fin n → Finset E},
  PlaysIn S σh →
    (∀ (τ' : Fin n → Finset E), PlaysIn S τ' → potential a σh ≤ potential a τ') →
      PlaysIn S τ → socialCost a σh ≤ 2 * socialCost a τ

-- Preuves sans sorry ni axiome ad hoc
#print axioms CongestionGames.potential_exchange
──────▶  'CongestionGames.potential_exchange' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms CongestionGames.minPotential_isNash
──────▶  'CongestionGames.minPotential_isNash' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms CongestionGames.cost_minPotential_le_two
──────▶  'CongestionGames.cost_minPotential_le_two' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 2

Raw input:
{"cmd": "-- Maillon 1 : x^2 <= x (x+1), arete par arete\n#check @socialCost_le_potential\n-- Maillon 2 : minimalite du potentiel sur les profils jouables\n#check @minPotential_isNash\n-- Maillon 3 : x (x+1) <= 2 x^2, arete par arete\n#check @potential_le_two_socialCost\n-- La chaine complete\n#check @cost_minPotential_le_two\n\n-- Preuves sans sorry ni axiome ad hoc\n#print axioms CongestionGames.potential_exchange\n#print axioms CongestionGames.minPotential_isNash\n#print axioms CongestionGames.cost_minPotential_le_two", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@socialCost_le_potential : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)\n  (σ : Fin n → Finset E), socialCost a σ ≤ potential a σ"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@minPotential_isNash : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)\n  (S : Fin n → Finset (Finset E)) {σh : Fin n → Finset E},\n  PlaysIn S σh → (∀ (τ : Fin n → Finset E), PlaysIn S τ → potential a σh ≤ potential a τ) → IsNash a S σh"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@potential_le_two_socialCost : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)\n  (σ : Fin n → Finset E), potential a σ ≤ 2 * socialCost a σ"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@cost_minPotential_le_two : ∀ {E : Type} [inst : Fintype E] [inst_1 : DecidableEq E] {n : ℕ} (a : E → ℕ)\n  (S : Fin n → Finset (Finset E)) {σh τ : Fin n → Finset E},\n  PlaysIn S σh →\n    (∀ (τ' : Fin n → Finset E), PlaysIn S τ' → potential a σh ≤ potential a τ') →\n      PlaysIn S τ → socialCost a σh ≤ 2 * socialCost a τ"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "'CongestionGames.potential_exchange' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"se

### Lecture du resultat

La chaine complete se lit :

$$C(\hat\sigma) \;\le\; \Phi'(\hat\sigma) \;\le\; \Phi'(\tau) \;\le\; 2\,C(\tau)$$

ou `σ̂` est l'equilibre **minimisant le potentiel** et `τ` n'importe quel profil jouable (l'optimum social en particulier) :

1. **`C ≤ Φ'`** : arete par arete, `x² ≤ x(x+1)` — le cout social est sous le potentiel double ;
2. **`Φ'(σ̂) ≤ Φ'(τ)`** : c'est la *definition* de `σ̂` ;
3. **`Φ' ≤ 2C`** : arete par arete, `x(x+1) ≤ 2x²` des que `x ≥ 1` (et egalite en 0).

Le premier `#print axioms` ne doit afficher que les axiomes standard de Lean (`propext`, `Classical.choice`, `Quot.sound`) — **aucun `sorryAx`**, aucun axiome ad hoc : la borne est un theoreme integrallement prouve, pas un enonce admis.

Comparez avec le carnet Python : la borne non-atomique 4/3 demande l'argument d'equalisation de Wardrop sur un *continuum* de flux ; la borne atomique 2x, elle, tient dans trois lignes d'inegalites sur `Nat` — le prix de la simplicite est un facteur plus gros.

## 4. Temoin 1 : l'anarchie atteint exactement 2

La borne 2x n'est pas un artefact de preuve : il existe une instance ou le **pire** equilibre de Nash coute exactement deux fois l'optimum. Le module la construit — 2 joueurs, 3 aretes, pentes `(2, 1, 3)`, avec des strategies **emboitees** : les deux joueurs peuvent prendre `e0` seul, et le joueur 2 peut combiner `e1` et `e2`.

In [4]:
open CongestionGames in
-- L'equilibre coordonne : P1 prend e1, P2 prend e0
#eval socialCost w2a w2Good
#eval potential w2a w2Good

-- L'equilibre anarchique : P1 prend e0, P2 emboite e1 + e2
#eval socialCost w2a w2Bad
#eval potential w2a w2Bad

-- Les deux sont des equilibres de Nash, et le rapport est exactement 2
#check @witness_ratio_exactly_two

open CongestionGames in
-- L'equilibre coordonne : P1 prend e1, P2 prend e0
#eval socialCost w2a w2Good
─────▶  3
#eval potential w2a w2Good
─────▶  6

-- L'equilibre anarchique : P1 prend e0, P2 emboite e1 + e2
#eval socialCost w2a w2Bad
─────▶  6
#eval potential w2a w2Bad
─────▶  12

-- Les deux sont des equilibres de Nash, et le rapport est exactement 2
#check @witness_ratio_exactly_two
──────▶  witness_ratio_exactly_two : IsNash w2a w2S w2Good ∧
  IsNash w2a w2S w2Bad ∧ socialCost w2a w2Bad = 2 * socialCost w2a w2Good
--% env 3

Raw input:
{"cmd": "open CongestionGames in\n-- L'equilibre coordonne : P1 prend e1, P2 prend e0\n#eval socialCost w2a w2Good\n#eval potential w2a w2Good\n\n-- L'equilibre anarchique : P1 prend e0, P2 emboite e1 + e2\n#eval socialCost w2a w2Bad\n#eval potential w2a w2Bad\n\n-- Les deux sont des equilibres de Nash, et le rapport est exactement 2\n#check @witness_ratio_exactly_two", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "3"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "12"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "witness_ratio_exactly_two : IsNash w2a w2S w2Good ∧\n  IsNash w2a w2S w2Bad ∧ socialCost w2a w2Bad = 2 * socialCost w2a w2Good"}],
 "env": 3}

### Lecture du resultat

Couts : 3 pour l'equilibre coordonne, 6 pour l'anarchique — rapport exactement **2**. Les deux profils sont des equilibres de Nash verifies par `decide` (enumeration des deviations disponibles). Notez les potentiels : 6 contre 12 — l'equilibre anarchique **n'est pas** potentiel-minimal, donc le theoreme de la section 3 ne dit rien de lui. C'est le temoin exact qui montre que son cout reste borne par 2 x optimum : la borne 2x est **atteinte**, mais pas depassee, parmi les equilibres de cette instance.

## 5. Temoin 2 : 15/7 — la borne ne couvre pas TOUS les equilibres

L'hypothese « equilibre potentiel-minimal » du theoreme n'est pas decorative. Sur cette instance a 3 joueurs (pentes `(3, 1, 3)`), il existe **deux** equilibres de Nash, et le pire coute 15 contre un optimum de 7 — strictement plus que deux fois l'optimum (14) :

In [5]:
open CongestionGames in
-- L'optimum (qui est aussi equilibre potentiel-minimal) : cout 7
#eval socialCost w3a w3Good
#eval potential w3a w3Good

-- Le pire equilibre de Nash : cout 15 > 2 x 7 = 14
#eval socialCost w3a w3Bad
#check @witness_beyond_two

open CongestionGames in
-- L'optimum (qui est aussi equilibre potentiel-minimal) : cout 7
#eval socialCost w3a w3Good
─────▶  7
#eval potential w3a w3Good
─────▶  14

-- Le pire equilibre de Nash : cout 15 > 2 x 7 = 14
#eval socialCost w3a w3Bad
─────▶  15
#check @witness_beyond_two
──────▶  witness_beyond_two : IsNash w3a w3S w3Good ∧ IsNash w3a w3S w3Bad ∧ 2 * socialCost w3a w3Good < socialCost w3a w3Bad
--% env 4

Raw input:
{"cmd": "open CongestionGames in\n-- L'optimum (qui est aussi equilibre potentiel-minimal) : cout 7\n#eval socialCost w3a w3Good\n#eval potential w3a w3Good\n\n-- Le pire equilibre de Nash : cout 15 > 2 x 7 = 14\n#eval socialCost w3a w3Bad\n#check @witness_beyond_two", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "7"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "14"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "15"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "witness_beyond_two : IsNash w3a w3S w3Good ∧ IsNash w3a w3S w3Bad ∧ 2 * socialCost w3a w3Good < socialCost w3a w3Bad"}],
 "env": 4}

### Lecture du resultat

Rapport 15/7 ≈ 2,14 > 2 : la borne elementaire echoue si on la lit comme « tout equilibre de Nash coute au plus 2 x optimum ». Le theoreme de la section 3 reste vrai — l'equilibre potentiel-minimal (cout 7 ici, l'optimum meme) est bien sous la borne — mais l'autre equilibre lui echappe.

La borne correcte pour **tous** les equilibres d'un jeu de congestion lineaire atomique est **5/2** (Christodoulou-Koutsoupias 2005), obtenue par l'argument de *lissegite* (smoothness) — un outil different, hors scope de ce module. Le temoin 15/7 montre deja que tout argument elementaire du type de la chaine de Rosenthal plafonne sous la verite.

## 6. Exercices

Trois exercices, chacun verifiable par enumeration `decide` une fois complete. Les enonces sont donnes en commentaire : a vous de retirer les commentaires et de completer.

In [6]:
-- Exercice 1 : deux aretes en PARALLELE, pentes egales -- l'anarchie y est nulle
-- Deux joueurs, strategies {e0} | {e1} pour chacun, pentes a = (1, 1).
-- TODO etudiant : montrer par enumeration que TOUT equilibre de Nash est
-- optimal sur cette instance. Conclusion visee : c'est l'emboitement des
-- strategies (temoin 1), pas la linearite seule, qui cree l'anarchie.

-- # Etape 1 : definir le jeu parallele
-- def pS : Fin 2 → Finset (Finset (Fin 2)) := ![{{0}, {1}}, {{0}, {1}}]
-- def pa : Fin 2 → ℕ := ![1, 1]

-- # Etape 2 : evaluer les 4 couts sociaux possibles
-- TODO etudiant : #eval socialCost pa ![{0}, {0}], ![{0}, {1}], ![{1}, {0}], ![{1}, {1}]

-- # Etape 3 : l'enonce final, prouvable par decide
-- theorem parallel_no_anarchy :
--     ∀ τ, IsNash pa pS τ → socialCost pa τ ≤ socialCost pa ![{0}, {1}] := by
--   decide [IsNash, socialCost, load]

#check @socialCost

-- Exercice 1 : deux aretes en PARALLELE, pentes egales -- l'anarchie y est nulle
-- Deux joueurs, strategies {e0} | {e1} pour chacun, pentes a = (1, 1).
-- TODO etudiant : montrer par enumeration que TOUT equilibre de Nash est
-- optimal sur cette instance. Conclusion visee : c'est l'emboitement des
-- strategies (temoin 1), pas la linearite seule, qui cree l'anarchie.

-- # Etape 1 : definir le jeu parallele
-- def pS : Fin 2 → Finset (Finset (Fin 2)) := ![{{0}, {1}}, {{0}, {1}}]
-- def pa : Fin 2 → ℕ := ![1, 1]

-- # Etape 2 : evaluer les 4 couts sociaux possibles
-- TODO etudiant : #eval socialCost pa ![{0}, {0}], ![{0}, {1}], ![{1}, {0}], ![{1}, {1}]

-- # Etape 3 : l'enonce final, prouvable par decide
-- theorem parallel_no_anarchy :
--     ∀ τ, IsNash pa pS τ → socialCost pa τ ≤ socialCost pa ![{0}, {1}] := by
--   decide [IsNash, socialCost, load]

#check @socialCost
──────▶  @socialCost : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ
--% env 5

Raw input:
{"cmd": "-- Exercice 1 : deux aretes en PARALLELE, pentes egales -- l'anarchie y est nulle\n-- Deux joueurs, strategies {e0} | {e1} pour chacun, pentes a = (1, 1).\n-- TODO etudiant : montrer par enumeration que TOUT equilibre de Nash est\n-- optimal sur cette instance. Conclusion visee : c'est l'emboitement des\n-- strategies (temoin 1), pas la linearite seule, qui cree l'anarchie.\n\n-- # Etape 1 : definir le jeu parallele\n-- def pS : Fin 2 \u2192 Finset (Finset (Fin 2)) := ![{{0}, {1}}, {{0}, {1}}]\n-- def pa : Fin 2 \u2192 \u2115 := ![1, 1]\n\n-- # Etape 2 : evaluer les 4 couts sociaux possibles\n-- TODO etudiant : #eval socialCost pa ![{0}, {0}], ![{0}, {1}], ![{1}, {0}], ![{1}, {1}]\n\n-- # Etape 3 : l'enonce final, prouvable par decide\n-- theorem parallel_no_anarchy :\n--     \u2200 \u03c4, IsNash pa pS \u03c4 \u2192 socialCost pa \u03c4 \u2264 socialCost pa ![{0}, {1}] := by\n--   decide [IsNash, socialCost, load]\n\n#check @socialCost", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 19, "column": 0},
   "endPos": {"line": 19, "column": 6},
   "data":
   "@socialCost : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ"}],
 "env": 5}

In [7]:
-- Exercice 2 : le mauvais equilibre du temoin 1 n'est PAS potentiel-minimal
-- TODO etudiant : verifier que Phi'(w2Bad) = 12 > Phi'(w2Good) = 6, puis
-- conclure en commentaire : le theoreme cost_minPotential_le_two ne dit
-- RIEN de w2Bad -- c'est le temoin exact (section 4) qui le borne.

-- # Etape 1 : evaluer les deux potentiels
-- #eval potential w2a w2Good
-- #eval potential w2a w2Bad

-- # Etape 2 : l'inegalite stricte par decide
-- example : potential w2a w2Good < potential w2a w2Bad := by
--   decide [potential, load]

#check @potential

-- Exercice 2 : le mauvais equilibre du temoin 1 n'est PAS potentiel-minimal
-- TODO etudiant : verifier que Phi'(w2Bad) = 12 > Phi'(w2Good) = 6, puis
-- conclure en commentaire : le theoreme cost_minPotential_le_two ne dit
-- RIEN de w2Bad -- c'est le temoin exact (section 4) qui le borne.

-- # Etape 1 : evaluer les deux potentiels
-- #eval potential w2a w2Good
-- #eval potential w2a w2Bad

-- # Etape 2 : l'inegalite stricte par decide
-- example : potential w2a w2Good < potential w2a w2Bad := by
--   decide [potential, load]

#check @potential
──────▶  @potential : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ
--% env 6

Raw input:
{"cmd": "-- Exercice 2 : le mauvais equilibre du temoin 1 n'est PAS potentiel-minimal\n-- TODO etudiant : verifier que Phi'(w2Bad) = 12 > Phi'(w2Good) = 6, puis\n-- conclure en commentaire : le theoreme cost_minPotential_le_two ne dit\n-- RIEN de w2Bad -- c'est le temoin exact (section 4) qui le borne.\n\n-- # Etape 1 : evaluer les deux potentiels\n-- #eval potential w2a w2Good\n-- #eval potential w2a w2Bad\n\n-- # Etape 2 : l'inegalite stricte par decide\n-- example : potential w2a w2Good < potential w2a w2Bad := by\n--   decide [potential, load]\n\n#check @potential", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 6},
   "data":
   "@potential : {E : Type} → [Fintype E] → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset E) → ℕ"}],
 "env": 6}

In [8]:
-- Exercice 3 : sur l'instance du temoin 1, TOUT equilibre respecte la borne 2
-- TODO etudiant : contrairement au temoin 2 (15/7), cette instance garde
-- tous ses equilibres sous 2 x optimum. Le prouver par enumeration.

-- theorem w2_all_nash_within_two :
--     ∀ τ, IsNash w2a w2S τ → socialCost w2a τ ≤ 2 * socialCost w2a w2Good := by
--   decide [IsNash, socialCost, load]

#check @IsNash

-- Exercice 3 : sur l'instance du temoin 1, TOUT equilibre respecte la borne 2
-- TODO etudiant : contrairement au temoin 2 (15/7), cette instance garde
-- tous ses equilibres sous 2 x optimum. Le prouver par enumeration.

-- theorem w2_all_nash_within_two :
--     ∀ τ, IsNash w2a w2S τ → socialCost w2a τ ≤ 2 * socialCost w2a w2Good := by
--   decide [IsNash, socialCost, load]

#check @IsNash
──────▶  @IsNash : {E : Type} → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset (Finset E)) → (Fin n → Finset E) → Prop
--% env 7

Raw input:
{"cmd": "-- Exercice 3 : sur l'instance du temoin 1, TOUT equilibre respecte la borne 2\n-- TODO etudiant : contrairement au temoin 2 (15/7), cette instance garde\n-- tous ses equilibres sous 2 x optimum. Le prouver par enumeration.\n\n-- theorem w2_all_nash_within_two :\n--     \u2200 \u03c4, IsNash w2a w2S \u03c4 \u2192 socialCost w2a \u03c4 \u2264 2 * socialCost w2a w2Good := by\n--   decide [IsNash, socialCost, load]\n\n#check @IsNash", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "@IsNash : {E : Type} → [DecidableEq E] → {n : ℕ} → (E → ℕ) → (Fin n → Finset (Finset E)) → (Fin n → Finset E) → Prop"}],
 "env": 7}

## Conclusion

- Le **potentiel double** `Phi' = sum a_e x_e (x_e + 1)` rend l'exactitude des deviations unilaterale entiere (`potential_exchange`) — c'est la que vit le facteur 2.
- La **chaine de Rosenthal** `C ≤ Phi' ≤ 2C` donne la borne elementaire 2x pour l'equilibre **potentiel-minimal** (`cost_minPotential_le_two`), sans fraction ni axiome.
- Le rapport 2 est **atteint** (temoin a 2 joueurs) ; il est **depasse** parmi les equilibres (temoin 15/7 a 3 joueurs) — la borne generale 5/2 (Christodoulou-Koutsoupias 2005) exige l'argument de lissegite, hors scope.

**References** : Rosenthal (1973), *A class of games possessing pure-strategy Nash equilibria*, Int. J. Game Theory 2 · Christodoulou & Koutsoupias (2005), *On the price of anarchy and stability of correlated equilibria of linear congestion games*, ESA 2005 · Roughgarden & Tardos (2002) pour la borne non-atomique 4/3, mesuree dans le [carnet Python 04h](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb).